Evaluation uses datasets, parameter values and fit records already produced
by the library. There is no single evaluation protocol: each comparison
requires the representation and assumptions of the helper being used.

## Which objects answer which questions?

| Question | Library inputs and operations | Complete procedure |
|---|---|---|
| Does the model reproduce a behavioral pattern? | Observed and simulated `RecallDataset` objects passed to the same analysis and plotting functions, such as `plot_spc()`. | [Simulation](../templates/simulation.ipynb) |
| What changes when a parameter changes? | Shared study inputs, a fixed model configuration and deliberate parameter interventions followed by simulation and analysis. | [Parameter exploration](../templates/parameter_shifting.ipynb) |
| How do likelihood fits compare? | Compatible named fit records passed to comparison helpers in `jaxcmr.summarize`. | [Model comparison](../templates/model_comparison.ipynb) |

A behavioral match, an intervention and a likelihood comparison answer
different questions. In particular, forced agreement with the observed
recall length is not evidence that a model predicts when recall stops.

## Fit-record conventions at the comparison boundary

`FitResult` provides parameter roles, objective values and row identifiers.
Comparison helpers additionally use a `name`. The templates record the data
selection, model configuration and objective needed to establish comparability.

Before comparing likelihood scores, establish that the records concern the
same observations, included events and likelihood definition. Check whether
parameters were pooled or independently fitted and how many values were
estimated. A lower discrepancy loss cannot be substituted for NLL merely
because both appear in a field named `fitness`.

The supplied source-only and broader eCMR records are pooled fits to the same
all-condition selection with the same termination-excluded objective and
stopping policy. Their emotional modulation differs in whether it also acts
on temporal context. Each has ten free parameters and one pooled fit row.

Both supplied eCMR fits learn before
context updating, fix emotional encoding drift at 1 and tie emotional recall
drift to temporal recall drift. They omit the legacy `emotion_drift_rate`.
Historical fits also require their original context equations and parameter
interpretations; a shared import path alone does not identify that behavior.

In [ ]:
#| code-summary: "0 — Locate the project inputs"

from pathlib import Path

from jaxcmr.helpers.paths import find_project_root, resolve_project_path

try:
    root_path = Path(find_project_root())
except FileNotFoundError:
    root_path = Path.cwd()


In [1]:
#| code-summary: "Read the named pooled records and their comparison basis"

from pathlib import Path
import json

files = {
    "Source-only eCMR": "Dupertuys2026_eCMR_source_only.json",
    "Broader eCMR": "Dupertuys2026_eCMR_broad.json",
}
records = [
    {**json.loads((Path(resolve_project_path("data/fits", root_path)) / filename).read_text()), "name": name}
    for name, filename in files.items()
]
for record in records:
    print(record["name"], "— fit IDs:", record["fits"]["subject"],
          "free parameters:", len(record["free"]))
print("Recorded trial selection:", records[0]["trial_query"])
print("Recorded loss:", records[0]["loss_function"])

Source-only eCMR — fit IDs: [-1] free parameters: 10
Broader eCMR — fit IDs: [-1] free parameters: 10
Recorded trial selection: data['list_type'] > 0
Recorded loss: jaxcmr.loss.transform_sequence_likelihood.ExcludeTerminationLikelihoodLoss


## Compare summed negative log likelihoods

`jaxcmr.summarize.calculate_aic()` accepts comparison records containing a model
`name`, top-level integer `subject` IDs, `free` parameter names, and `fitness`.
It returns a DataFrame sorted by AIC. Each `fitness` entry must be a summed
negative log likelihood for one fitted parameter set:

$$\mathrm{AIC} = 2k + 2\,\mathrm{NLL}.$$

The saved fits already contain NLL. Select their scores and IDs explicitly and
use `list(record["free"])` to supply parameter names without optimizer bounds.
Keep the original fit records for parameter summaries and simulation.

A pooled fit has `subject=[-1]` and incurs one parameter penalty. Independently
fitted participant rows incur one penalty each. Supply unique integer IDs in
`subject`, aligned with the scores and identifying the same set across models.
Nested `fits["subject"]`, plural `subjects`, omitted IDs, and string or floating-point
IDs are not alternative comparison inputs. These are caller requirements;
comparison functions do not normalize or validate those alternative forms.


In [2]:
#| code-summary: "Prepare comparison records and calculate AIC"

from jaxcmr.summarize import ModelComparison, calculate_aic

comparison_records: list[ModelComparison] = [
    {
        "name": record["name"],
        "subject": record["fits"]["subject"],
        "fitness": record["fitness"],
        "free": list(record["free"]),
    }
    for record in records
]
calculate_aic(comparison_records)

,Model,AIC
1,Broader eCMR,14348.715746
0,Source-only eCMR,14358.852540


The supplied source-only and broader fits have NLLs 7169.426 and 7164.358,
respectively. Their AIC difference is 10.137, favoring broader eCMR.
The within-model differences between the two three-restart seed sets are
0.005 and 0.049 NLL units.
The ranking is the same in both seed sets.

The result is a library-produced table. Lower AIC is preferred within this
comparison; it does not establish a unique explanation of the experiment.
Each model contributes one pooled parameter set, so `len(free)` is the
parameter count for that row. `calculate_aic_weights()` uses the same NLL
convention and parameter penalties as `calculate_aic()`.

Participant-wise comparisons align rows by subject identifier and require
finite scores for the same participant set. A pooled row is not a sample
of independent participant estimates. Mean-difference helpers return the
mean and interval half-width; median-difference helpers return the median
and the actual bootstrap lower and upper endpoints.

`calculate_bic_scores(results, n_obs)` and the AICc helpers require positive
observation counts for each fitted parameter row. Counts follow the first
record's original participant order; the library aligns them with the scores.
Specify the scientific counting unit and selection explicitly. A scalar count
is valid only for one fitted row; AICc requires each count to exceed `k + 1`.

`floor_nested_fitness(comparison_records, nesting_pairs)` returns separate comparison
records and an adjustment table without changing the raw fits. Label adjusted
scores and save the adjustment table beside them. Keep raw fits for parameter
summaries, simulations and held-out evaluation: an adjusted score is not a newly
optimized parameter set. The helper trusts the declared nesting relationships.
Supply pairs of model names, such as `[("CMR", "eCMR")]`; two-element lists from
JSON or YAML also work. Stringified Python expressions are not parsed.

Bayesian model selection uses an AIC/AICc-based evidence approximation over
participant fits. A pooled row cannot supply participant-level inference;
failure to converge raises an error rather than returning a result.

## Experimental cross-validation functionality

`jaxcmr.experimental.cross_validation` contains an experimental driver, with
`CVResult` describing fold-specific fits, participant identifiers, training
scores and held-out scores. The current driver mixes pooled fitting with
participant-level scoring and can return incorrect held-out scores; it is
not ready for scientific use. A held-out
comparison requires an appropriate split, fitting only the training
observations and evaluating the held-out observations under the chosen
objective. It is a different procedure from comparing in-sample AIC values.

For held-out comparisons, `CVComparison` uses the same top-level integer `subject`
field and keeps held-out objectives in `cv_fitness`. Given an independently verified
`cv_result`, the input is `{"name": name, "subject": cv_result["subjects"],
"cv_fitness": cv_result["cv_fitness"]}`. This field selection does not resolve the
experimental driver's scientific limitations.

See [Fitting Procedures](fitting.ipynb) for result structure and the
[model comparison template](../templates/model_comparison.ipynb) for the full
pooled comparison. These interfaces organize the evidence; selecting a
scientifically meaningful comparison remains part of the research design.